## TODO 2: Build the full Separator module

You now have both pieces — `Encoder` and `TCNBlock`. The `Separator` wraps everything in between: it takes the encoder output, runs it through the full TCN stack, and produces masks.

Looking back at the paper figure you screenshotted, the separator does:

```
H: (batch, 512, L)          ← encoder output
    ↓
LayerNorm + 1×1 Conv         ← normalize + compress 512 → 128
    ↓
TCNs (8 blocks × 3 repeats)  ← 24 total TCNBlock calls
    ↓
1×1 Conv + Sigmoid           ← expand 128 → 512*C, reshape to C masks
    ↓
masks: (batch, C, 512, L)    ← one mask per speaker, values in [0,1]
```

**Parameters from the paper:**
- `N = 512` — encoder filters (input channels to separator)
- `B = 128` — bottleneck dimension (what TCN operates at)
- `num_blocks = 8` — dilations per repeat: `[1, 2, 4, 8, 16, 32, 64, 128]`
- `num_repeats = 3` — how many times to repeat the 8-block stack
- `num_speakers = 2` — number of masks to produce (C)

**Algorithm in plain English:**
```
1. GroupNorm(1, N) on input H

2. 1×1 Conv: N → B  (bottleneck compression)

3. Loop num_repeats times:
       Loop over dilations [1, 2, 4, 8, 16, 32, 64, 128]:
           pass through TCNBlock(in_channels=B, dilation=d)

4. 1×1 Conv: B → N * num_speakers  (expand for mask generation)

5. Sigmoid activation

6. Reshape: (batch, N*C, L) → (batch, C, N, L)
```

**Requirements:**
- `nn.Module` subclass
- Parameters: `N=512, B=128, num_blocks=8, num_repeats=3, num_speakers=2`
- Build the TCN stack in `__init__` using `nn.ModuleList` — not a plain Python list
- Output shape: `(batch, num_speakers, N, L)`

**Hints:**
- `nn.ModuleList` is like a Python list but PyTorch can see its parameters — if you use a regular list, `.parameters()` won't find the TCN weights and they won't train
- For step 6, `.view(batch, C, N, L)` or `.reshape(batch, C, N, L)` — think about which dimension you're splitting
- The dilations for `num_blocks=8` are always powers of 2: `[2**i for i in range(num_blocks)]`
- `nn.Sequential` is another option for the TCN stack — but `nn.ModuleList` gives you more flexibility for later when EEND-SS needs to extract intermediate features

**Expected shapes:**
```python
separator = Separator(N=512, B=128, num_speakers=2)
H = torch.randn(4, 512, 59999)   # encoder output

masks = separator(H)
assert masks.shape == (4, 2, 512, 59999)   # batch, speakers, filters, frames

separator3 = Separator(N=512, B=128, num_speakers=3)
masks3 = separator3(H)
assert masks3.shape == (4, 3, 512, 59999)
```

**After it works — question to answer:**

Count the total number of `TCNBlock` instances created inside `Separator`. Then print `separator.parameters()` count. How many of those parameters belong to the TCN blocks vs the 1×1 convolutions? Which part of the separator dominates the parameter count?

Go implement — come back with code and your answer.

In [1]:
import torch
from torch import nn

from pathlib import Path
import os
import sys

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent
sys.path.insert(0, str(project_root))

from src.data.loaders import get_dataloaders
from src.models.conv_tasnet import _TCN


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

✅ Using Apple Silicon GPU
PyTorch version: 2.1.0
Device: mps


# todo 2

In [2]:
class Separator(nn.Module):
    def __init__(
            self,
            N: int = 512,    # encoder filters
            B: int = 128,    # bottleneck dimension
            num_blocks = 8,  # dilations per repeat [1, 2, 4, 8, 16, 32, 64, 128]
            num_repeats = 3, # how many times to repeat the 8-block stack
            num_speakers = 2 # number of masks to produce (C)
            ):
        super().__init__()
        
        self.in_channels = N
        self.Bottleneck = B
        self.dilations = [2**i for i in range(num_blocks)]
        self.num_repeats = num_repeats
        self.num_speakers = num_speakers
        
        self.groupNorm = nn.GroupNorm(1, self.in_channels)
        self.bottleneckCompression = nn.Conv1d(
            in_channels=self.in_channels, 
            out_channels=self.Bottleneck, 
            kernel_size=1, 
            bias = False
            )
        self.tcnStack = nn.ModuleList()
        
        for _ in range(self.num_repeats):
            for dilation in self.dilations:
                self.tcnStack.append(_TCN(
                    in_channels=self.Bottleneck, 
                    dilation=dilation
                    ))
        
        self.bottleneckExpension = nn.Conv1d(
            in_channels=self.Bottleneck, 
            out_channels=self.in_channels*self.num_speakers, 
            kernel_size=1, 
            bias = False
            )
        self.activation = nn.Sigmoid()
    
    def forward(self, x):
        out = self.groupNorm(x)
        out = self.bottleneckCompression(out)
        for tcn in self.tcnStack:
            out = tcn(out)
        out = self.bottleneckExpension(out)
        out = self.activation(out)
        b, _, L = out.shape
        out = out.view(b, self.num_speakers, self.in_channels, L)
        return out


In [3]:
separator = Separator(N=512, B=128, num_speakers=2).to(device=device)
H = torch.randn(4, 512, 59999).to(device=device)   # encoder output

masks = separator(H)
assert masks.shape == (4, 2, 512, 59999)   # batch, speakers, filters, frames

separator3 = Separator(N=512, B=128, num_speakers=3).to(device=device)
masks3 = separator3(H)
assert masks3.shape == (4, 3, 512, 59999)

RuntimeError: MPS backend out of memory (MPS allocated: 8.36 GB, other allocations: 706.15 MB, max allowed: 9.07 GB). Tried to allocate 117.19 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
for name, param in separator.named_parameters():
    print(f"Layer: {name} | Shape: {param.shape} | Requires Grad: {param.requires_grad}")